In [1]:
from strategies import *
from scipy.signal import savgol_filter
import pandas as pd

if __name__ == '__main__':
    
    df = pd.read_csv("../../backtesting/data/GOOGL/GOOGL.USUSD_Candlestick_5_M_ASK_05.10.2022-05.10.2024.csv")

    df['Gmt time']=df["Gmt time"].str.replace(".000","")
    df['Gmt time']=pd.to_datetime(df['Gmt time'],format='%d.%m.%Y %H:%M:%S')
    datetime_est=pd.to_datetime(df["Gmt time"], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern')
    
    fromTodayStart = '2022-10-05 09:30:00'
    toNow   = '2022-12-05 16:00:00'
    df = df[datetime_est.between(fromTodayStart, toNow)].copy()

    df.rename(columns={"Open": "open"}, inplace=True)
    df.rename(columns={"High": "high"}, inplace=True)
    df.rename(columns={"Low": "low"}, inplace=True)
    df.rename(columns={"Volume": "volume"}, inplace=True)
    df.rename(columns={"Close": "close"}, inplace=True)
    df.rename(columns={"Gmt time": "time"}, inplace=True)

    df = df[df.notnull().all(axis=1)]
    df=df[(df.volume != 0)]
    df=df[df.high!=df.low]
    
    df.reset_index(drop=True, inplace=True)
    df['index']=df.index
    
    df.set_index("time", inplace=True, drop=True)
    
    df = df[['index','high','low','close','volume','open']] 

    result = squeez(df)

    log_sequence = []
    
    for candle in range(0, len(result)):
        
        position = (result.iloc[candle]).position
        date = (result.iloc[candle]).date_est
        time = (result.iloc[candle]).time_est
        buying_price = (result.iloc[candle]).buying_price
        selling_price = (result.iloc[candle]).selling_price
        close_smooth = (result.iloc[candle]).close_smooth
        close = (result.iloc[candle]).close
        squeezeCloseToEma = (result.iloc[candle]).squeezeCloseToEma
        squeezedArea = (result.iloc[candle]).squeezedArea
        ema25 = (result.iloc[candle]).ema25
        
        if (position==1):
            trans = [date, time, candle, 'GOOGLE', 10, round(buying_price, 2), close, close_smooth, squeezedArea, squeezeCloseToEma, 'B']
        elif (position==-1):  
            trans = [date, time, candle, 'GOOGLE', 10, round(selling_price, 2), close, close_smooth, squeezedArea, squeezeCloseToEma,'S']
        else:
            trans = [date, time, candle, 'GOOGLE', 10, 0, close, close_smooth, squeezedArea, squeezeCloseToEma,'N']
            
        log_sequence.append(trans)

    #df_log_sequence = pd.DataFrame(log_sequence, columns=['Date','Time','Candle','Symbol','Quantity','Price', 'Close', 'Close Smooth','Squeez' ,'SCE', 'Side'])
    #df_log_sequence.to_csv('live_test_script.csv', index=False)
    
    result.to_csv('live_test_script.csv', index=False)

In [ ]:
def exitArea(x):
    extraSpace = 1.20
    if (x['position']==POSITION['NEUTRAL'].value):
        return x['high']+extraSpace    
        
def positionArea(x):
    extraSpace = 1.20
    if (x['position']==POSITION['LONG'].value):
        return x['low']-extraSpace
    elif (x['position']==POSITION['SHORT'].value):
        return x['high']+extraSpace    
    else:
        return np.nan
                
def squeezeCloseToEmaArea(x):
    extraSpace = .80
    if (x['squeezeCloseToEma']==EMA25['PRICE_ACCION_OVER_EMA'].value):
        return x['low']-extraSpace
    elif (x['squeezeCloseToEma']==EMA25['PRICE_ACCION_UNDER_EMA'].value):
        return x['high']+extraSpace
    else:
        return np.nan
        
def squezzeArea(x):
    extraSpace = .50
    if (x['squeezedArea']==EMA25['PRICE_ACCION_OVER_EMA'].value):
        return x['low']-extraSpace
    elif (x['squeezedArea']==EMA25['PRICE_ACCION_UNDER_EMA'].value):
        return x['high']+extraSpace
    else:
        return np.nan
        
with pd.option_context("mode.copy_on_write", True): 
    result['squezzeArea'] = result.apply(lambda row: squezzeArea(row), axis=1)
    result['squeezeCloseToEmaArea'] = result.apply(lambda row: squeezeCloseToEmaArea(row), axis=1)
    result['positionArea'] = result.apply(lambda row: positionArea(row), axis=1)
    result['exitArea'] = result.apply(lambda row: exitArea(row), axis=1)

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go  

dfpl = result[0:100]

fig = go.Figure(
    data=[
        go.Candlestick(
            x=dfpl['index'], 
            open=dfpl['open'],
            high=dfpl['high'],
            low=dfpl['low'],
            close=dfpl['close']),
        go.Scatter(
            x=dfpl['index'], 
            y=dfpl['ema25'],
            line=dict(color='orange', width=1), 
            name="EMA 25"),
        go.Scatter(
            x=dfpl['index'], 
            y=dfpl['close_smooth'],
            line=dict(color='blue', width=1), 
            name="Close Smooth")
    ]
)

fig.add_scatter(x=dfpl['index'], y=dfpl['squezzeArea'], mode="markers", marker=dict(size=5, color="red"), name="Squeez")
fig.add_scatter(x=dfpl['index'], y=dfpl['squeezeCloseToEmaArea'], mode="markers", marker=dict(size=5, color="orange"), name="Important area")
fig.add_scatter(x=dfpl['index'], y=dfpl['positionArea'], mode="markers", marker=dict(size=5, color="green"), name="Entry")
fig.add_scatter(x=dfpl['index'], y=dfpl['exitArea'], mode="markers", marker=dict(size=5, color="yellow"), name="Exit")

fig.update_layout(xaxis_rangeslider_visible=False)
fig.show()